# Annotation Confidence v2 GBM — Applied to Orbitrap HILIC posESI

**Goal:** apply the neg-trained GBM from `annotation_confidence_v2.ipynb` to pos data.

**Data:**
- Pos annotations: 3,738 (TP=3,024, FP=714)
- Pos hits: from `data/library_hits/orbitrap_pos_fetch_cache.json` (has library_wiki_id)
- Pos query peaks: `data/pos_query_peaks_cache.json` (fetched for this task)
- Library peaks: `data/library_peaks_cache.json` (shared with neg, 27K cached)

**Features:** Same 15 as v2 (delta_rt_abs, spectral_entropy, anno_entropy_sim, anno_forward, anno_reverse, anno_delta_mda, sim_gap, anno_rank, max_deviation, is_isf_adduct, isf_no_mh, is_dubious_adduct, n_compound_adducts, is_nacetyl).

**Adduct override for pos:** `M+H` flipped from `dubious` → `ok` (same alphabetical sort artifact Oliver flagged for `M-H`).

**Pipeline:** retrain neg GBM inline, freeze, apply to pos, report AUC.

In [ ]:
import os, json, re, time
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
import ms_entropy
from rdkit import Chem, RDLogger
from rdkit.Chem.inchi import MolToInchi, InchiToInchiKey
RDLogger.DisableLog('rdApp.*')
import matplotlib.pyplot as plt

ROOT = '/Users/ellayoung/Desktop/metabolo_confi_score'
NEG_XLSX       = f'{ROOT}/data/masswiki_Orbitrap HILIC negESI_2026-03-19.xlsx'
NEG_HITS_CSV   = f'{ROOT}/data/orbitrap_hits_refetched.csv'
POS_CSV        = f'{ROOT}/data/Orbitrap_HILIC_posESI_curated_041326.csv'
POS_FETCH_CACHE = f'{ROOT}/data/library_hits/orbitrap_pos_fetch_cache.json'
LIB_PEAKS      = f'{ROOT}/data/library_peaks_cache.json'
POS_QUERY_PEAKS = f'{ROOT}/data/pos_query_peaks_cache.json'
QUERY_PEAKS    = f'{ROOT}/data/query_peaks_cache.json'
ADDUCT_TAX     = f'{ROOT}/data/adduct_taxonomy_oliver.csv'
SOLID_TP_PATH  = f'{ROOT}/data/solid_tp.csv'
INCHIKEY_CACHE = f'{ROOT}/data/inchikey_cache.json'
OUT_DIR        = f'{ROOT}/results/orbitrap_pos_gbm'
os.makedirs(OUT_DIR, exist_ok=True)

PPM_TOL = 10.0  # match v2

# InChIKey cache (shared)
ik_cache = {}
if os.path.exists(INCHIKEY_CACHE):
    with open(INCHIKEY_CACHE) as f:
        ik_cache = json.load(f)

def get_ik14(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return ''
    if smiles in ik_cache:
        return ik_cache[smiles].get('ik14', '')
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ''
    ik = InchiToInchiKey(MolToInchi(mol))
    if ik:
        ik_cache[smiles] = {'ik14': ik[:14], 'inchikey': ik}
        return ik[:14]
    return ''

print('Setup complete')

## 1. Adduct Taxonomy (with M+H override)

In [ ]:
adduct_tax = pd.read_csv(ADDUCT_TAX)
adduct_lookup = dict(zip(adduct_tax['adduct'].str.strip(), adduct_tax['category']))

# Overrides: M+H/M-H alphabetical artifact + pos-mode ok adducts
for a in ['M+H', 'M-H', 'M+Na', 'M+NH4', 'M+K', '2M+H', '2M+Na', '2M+K', '2M+NH4']:
    old = adduct_lookup.get(a, 'NOT FOUND')
    adduct_lookup[a] = 'ok'
    print(f'  Override: {a!r}  {old} -> ok')

def norm_adduct(s):
    if not isinstance(s, str): return ''
    s = s.strip()
    s = re.sub(r'^\[', '', s)
    s = re.sub(r'\][\+\-]?\d*[\+\-]?$', '', s)
    s = re.sub(r'[\+\-]$', '', s)
    return s.strip()

def classify_adduct(a):
    norm = norm_adduct(a)
    cat = adduct_lookup.get(norm)
    if cat: return cat
    # Rule-based ISF: M+H-X (pos) or M-H-X (neg) = neutral loss
    if re.match(r'M\+H-', norm) or re.match(r'M-H-', norm):
        return 'isf'
    return 'unknown'

for test in ['[M+H]+', '[M+Na]+', '[M+NH4]+', '[M+H-H2O]+', '[M+H-NH3]+', '[M+K]+', '[2M+H]+']:
    print(f'  {test!r} -> {classify_adduct(test)}')

## 2. Load & Label Pos Data

In [ ]:
pos_raw = pd.read_csv(POS_CSV, low_memory=False)
print(f'Pos spectra: {len(pos_raw):,}')

# Labels
names = pos_raw['name'].fillna('').astype(str)
yy = names.str.startswith('yy_')
zz = names.str.startswith('zz_')
pos_raw['label'] = 'unlabeled'
pos_raw.loc[pos_raw['is_manual_annotated'].astype(bool) & ~yy & ~zz & (names.str.strip() != ''), 'label'] = 'TP'
pos_raw.loc[yy, 'label'] = 'FP'
pos_raw.loc[zz, 'label'] = 'TN'

# For pos, use simple tiers: all TPs are "regular_tp" since we don't have color tiers here
pos_raw['tier'] = 'unlabeled'
pos_raw.loc[pos_raw['label']=='TP', 'tier'] = 'regular_tp'
pos_raw.loc[pos_raw['label']=='FP', 'tier'] = 'fp'

pos = pos_raw[pos_raw['label'].isin(['TP','FP'])].copy().reset_index(drop=True)
print(f'\nPos annotated (TP+FP): {len(pos):,}')
print(pos['label'].value_counts().to_string())

In [ ]:
# ── Adduct features on pos ──
pos['adduct_cat'] = pos['adduct'].apply(classify_adduct)
pos['is_isf_adduct'] = (pos['adduct_cat']=='isf').astype(int)
pos['is_dubious_adduct'] = (pos['adduct_cat']=='dubious').astype(int)

pos['name_lower'] = pos['name'].fillna('').str.strip().str.lower()
has_ok = (pos[pos['adduct_cat']=='ok']
          .groupby('name_lower')['wiki_id'].count().rename('_has_ok'))
pos = pos.merge(has_ok.reset_index(), on='name_lower', how='left')
pos['has_ok_adduct'] = pos['_has_ok'].fillna(0).clip(upper=1).astype(int)
pos = pos.drop(columns='_has_ok')
pos['isf_no_mh'] = ((pos['is_isf_adduct']==1) & (pos['has_ok_adduct']==0)).astype(int)

n_add = pos.groupby('name_lower')['adduct'].nunique().rename('n_compound_adducts')
pos = pos.merge(n_add.reset_index(), on='name_lower', how='left')

pos['is_nacetyl'] = pos['name'].fillna('').str.contains(
    r'(?i)^N-?acetyl|^N\d-acetyl|^acetyl-.*(?:amine|alanine|valine|leucine|'
    r'isoleucine|glycine|serine|threonine|cysteine|methionine|phenylalanine|'
    r'tyrosine|tryptophan|aspart|glutam|histid|lysine|arginine|proline|ornithine|'
    r'citrulline|carnosine)'
).astype(int)

print('=== Pos adduct features ===')
print(pos['adduct_cat'].value_counts().to_string())
print(f"\nis_isf_adduct: {pos['is_isf_adduct'].sum()}, is_dubious_adduct: {pos['is_dubious_adduct'].sum()}")
print(f"has_ok_adduct: {pos['has_ok_adduct'].sum()}, isf_no_mh: {pos['isf_no_mh'].sum()}")
print(f"is_nacetyl: {pos['is_nacetyl'].sum()}")

## 3. Load Pos Hits from Fetch Cache (with library_wiki_id)

In [ ]:
with open(POS_FETCH_CACHE) as f:
    pos_cache = json.load(f)
print(f'Pos cache: {len(pos_cache):,} entries')

# Flatten REF hits with library_wiki_id, for pos annotated only
pos_wids = set(pos['wiki_id'])
hit_rows = []
for wid in pos_wids:
    bundle = pos_cache.get(wid)
    if not bundle:
        continue
    for h in (bundle.get('ref') or []):
        hit_rows.append({
            'wiki_id': wid,
            'library_wiki_id': h.get('library_wiki_id'),
            'db': h.get('db') or h.get('source'),
            'id': h.get('id'),
            'lib_name': h.get('name'),
            'smiles': h.get('smiles'),
            'adduct': h.get('adduct'),
            'lib_precursor_mz': h.get('precursor_mz') or h.get('precursor'),
            'entropy_similarity': h.get('entropy_similarity') or h.get('score'),
            'rank': h.get('rank'),
        })
hits = pd.DataFrame(hit_rows)
print(f'Ref hits for pos annotated: {len(hits):,} rows, {hits["wiki_id"].nunique():,} spectra')

In [ ]:
# Compute IK14 for hits + annotations
print('Computing InChIKey14 for hits...')
hits['hit_ik14'] = hits['smiles'].fillna('').apply(get_ik14)
# for annotations: use annotation-smiles
pos['anno_ik14'] = pos['annotation-smiles'].fillna('').apply(get_ik14)

with open(INCHIKEY_CACHE, 'w') as f:
    json.dump(ik_cache, f)
print(f'  hit_ik14 coverage: {(hits["hit_ik14"]!="").sum()}/{len(hits)}')
print(f'  anno_ik14 coverage: {(pos["anno_ik14"]!="").sum()}/{len(pos)}')

# Join hits with pos label/ik14
named_info = pos[['wiki_id','anno_ik14','label','tier']].copy()
joint = hits.merge(named_info, on='wiki_id', how='inner')
joint['is_anno_hit'] = (joint['hit_ik14']!='') & (joint['anno_ik14']!='') & (joint['hit_ik14']==joint['anno_ik14'])
print(f'Joint: {len(joint):,}, annotation hits (IK14 match): {joint["is_anno_hit"].sum():,}')

# Dedup by (wiki_id, hit_ik14), keep max entropy_similarity
joint['entropy_similarity'] = pd.to_numeric(joint['entropy_similarity'], errors='coerce')
joint['dedup_key'] = joint.apply(
    lambda r: (r['wiki_id'], r['hit_ik14']) if r['hit_ik14'] else (r['wiki_id'], r['lib_name']),
    axis=1)
before = len(joint)
joint = joint.sort_values('entropy_similarity', ascending=False).drop_duplicates('dedup_key').reset_index(drop=True)
print(f'After dedup: {before} -> {len(joint)}')

## 4. Fetch Missing Library Peaks (no auth)

In [ ]:
with open(LIB_PEAKS) as f:
    lib_peaks_cache = json.load(f)
print(f'Library peaks cache: {len(lib_peaks_cache):,}')

# Find missing library_wiki_ids referenced by pos hits
lwids = joint['library_wiki_id'].dropna().unique().tolist()
missing = [l for l in lwids if l not in lib_peaks_cache]
print(f'Pos hits reference {len(lwids):,} library entries; {len(lwids)-len(missing):,} cached, {len(missing):,} missing')

if missing:
    import requests
    LIB_URL = 'https://masswiki.us-west-2.elasticbeanstalk.com/reference_library/get_spectra_data'
    print(f'Fetching {len(missing)} missing library peaks...')
    errors = 0
    for i in range(0, len(missing), 50):
        batch = missing[i:i+50]
        try:
            r = requests.post(LIB_URL,
                              json={'id_list': batch, 'get_details': False, 'include_fields': ['peaks']},
                              timeout=30)
            if r.status_code == 200:
                for e in r.json():
                    wid = e.get('wiki_id')
                    pk  = e.get('peaks', [])
                    if wid and pk:
                        lib_peaks_cache[wid] = pk
                    elif wid:
                        errors += 1
        except Exception as ex:
            errors += 1
        time.sleep(0.1)
    with open(LIB_PEAKS, 'w') as f:
        json.dump(lib_peaks_cache, f)
    print(f'  Done. Cache: {len(lib_peaks_cache):,}. Errors: {errors}')

# Load pos query peaks
with open(POS_QUERY_PEAKS) as f:
    pos_query_peaks = json.load(f)
print(f'Pos query peaks: {len(pos_query_peaks):,}')

## 5. Compute Features (per spectrum)

In [ ]:
def compute_scores(q_peaks, l_peaks, ppm_tol=PPM_TOL):
    if not q_peaks or not l_peaks:
        return None
    try:
        q_c = ms_entropy.clean_spectrum(q_peaks)
        l_c = ms_entropy.clean_spectrum(l_peaks)
        q_w = ms_entropy.apply_weight_to_intensity(q_c)
        l_w = ms_entropy.apply_weight_to_intensity(l_c)
        q_a = np.array(q_w, dtype=float)
        l_a = np.array(l_w, dtype=float)
        if len(q_a)==0 or len(l_a)==0: return None
        q_mz,q_i = q_a[:,0], q_a[:,1]
        l_mz,l_i = l_a[:,0], l_a[:,1]
        q_n = q_i/q_i.sum(); l_n = l_i/l_i.sum()
        mL=mQ=0.0; pairs=[]; used_q = np.zeros(len(q_mz), bool)
        for j,(lm,li) in enumerate(zip(l_mz,l_n)):
            tol = lm * ppm_tol / 1e6
            d = np.abs(q_mz - lm)
            cand = np.where((d<=tol) & ~used_q)[0]
            if len(cand):
                b = cand[np.argmin(d[cand])]
                mL += li; mQ += q_n[b]
                pairs.append((q_i[b], l_i[j]))
                used_q[b] = True
        res = {'reverse_score':mL, 'forward_score':mQ, 'n_matched':len(pairs)}
        if len(pairs) >= 2:
            qa = np.array([p[0] for p in pairs])
            la = np.array([p[1] for p in pairs])
            res['max_deviation'] = np.max(np.abs(np.log2((qa+1e-6)/(la+1e-6))))
        else:
            res['max_deviation'] = np.nan
        return res
    except Exception:
        return None

print('compute_scores ready')

In [ ]:
# Build per-spectrum annotation hit features
t0 = time.time()
rows = []
for wid, g in joint.groupby('wiki_id'):
    anno = g[g['is_anno_hit']]
    others = g[~g['is_anno_hit']]
    if len(anno) > 0:
        best = anno.loc[anno['entropy_similarity'].idxmax()]
        next_sim = others['entropy_similarity'].max() if len(others) > 0 else 0.0
        anno_rank = int((g['entropy_similarity'] >= best['entropy_similarity']).sum())
        lib_pk = lib_peaks_cache.get(best.get('library_wiki_id',''))
        qry_pk = pos_query_peaks.get(wid)
        sc = compute_scores(qry_pk, lib_pk)
        lib_mz = pd.to_numeric(best.get('lib_precursor_mz'), errors='coerce')
        obs_mz = pos.loc[pos['wiki_id']==wid,'precursor_mz'].iloc[0]
        row = {'wiki_id': wid,
               'anno_entropy_sim': best['entropy_similarity'],
               'anno_delta_mda': abs(obs_mz-lib_mz)*1000 if pd.notna(lib_mz) and lib_mz>0 else np.nan,
               'sim_gap': max(0.0, best['entropy_similarity']-next_sim) if pd.notna(next_sim) else best['entropy_similarity'],
               'anno_rank': anno_rank,
               'has_anno_hit': True}
        if sc:
            row['anno_reverse']=sc['reverse_score']; row['anno_forward']=sc['forward_score']; row['max_deviation']=sc['max_deviation']
        else:
            row['anno_reverse']=np.nan; row['anno_forward']=np.nan; row['max_deviation']=np.nan
    else:
        row = dict.fromkeys(['anno_entropy_sim','anno_delta_mda','sim_gap','anno_rank',
                              'anno_reverse','anno_forward','max_deviation'], np.nan)
        row['wiki_id']=wid; row['has_anno_hit']=False
    rows.append(row)
anno_hit_df = pd.DataFrame(rows)
print(f'Feature compute: {len(anno_hit_df):,} spectra in {time.time()-t0:.1f}s')
print(f'  With anno hit: {anno_hit_df["has_anno_hit"].sum():,}')

# Evidence table
ev = pos[['wiki_id','name','label','tier','anno_ik14','precursor_mz',
           'is_isf_adduct','is_dubious_adduct','has_ok_adduct','isf_no_mh',
           'n_compound_adducts','is_nacetyl','adduct']].copy()
ev['spectral_entropy'] = pd.to_numeric(pos['entropy'], errors='coerce')
ev['delta_rt_abs'] = pd.to_numeric(pos['anno_delta_rt'], errors='coerce').abs()
ev['identity_score'] = pd.to_numeric(pos['identity_score'], errors='coerce')
ev = ev.merge(anno_hit_df, on='wiki_id', how='left')
print(f'\nEvidence table: {len(ev):,} rows')

for ch in ['delta_rt_abs','spectral_entropy','anno_entropy_sim','anno_forward','anno_reverse',
           'anno_delta_mda','sim_gap','anno_rank','max_deviation']:
    n = ev[ch].notna().sum()
    print(f'  {ch:20s}  {n:>5d}/{len(ev)} ({n/len(ev):.1%})')

## 6. Train Neg GBM (reproduce v2)

Load neg data and features to retrain the identical GBM, then freeze for pos inference.

In [ ]:
# Load neg spectra + apply same feature pipeline
neg_raw = pd.read_excel(NEG_XLSX, header=4)
neg_raw['label'] = 'unlabeled'
neg_raw.loc[:1297, 'label'] = 'TP'
neg_raw.loc[neg_raw['name'].str.startswith('yy_', na=False), 'label'] = 'FP'
neg_raw.loc[neg_raw['name'].str.startswith('zz_', na=False), 'label'] = 'TN'

# solid_tp is uncertain tier (user correction 2026-04-13): treat as golden_tp holdout
solid = pd.read_csv(SOLID_TP_PATH)
solid_wids = set(solid['wiki_id'])

neg_raw['tier'] = 'first_pass'
# v2 also combines color-coded golden (green) but we don't recompute colors here; approximate:
# - rows 0-1297 with yy_ -> fp
# - rows 0-1297 solid_tp -> golden (holdout)
# - rows 0-1297 otherwise -> regular_tp
neg_raw.loc[:1297, 'tier'] = 'regular_tp'
neg_raw.loc[neg_raw['wiki_id'].isin(solid_wids), 'tier'] = 'golden_tp'
neg_raw.loc[neg_raw['label']=='FP', 'tier'] = 'fp'

neg = neg_raw[neg_raw['label'].isin(['TP','FP'])].copy().reset_index(drop=True)

# Adduct features (uses same taxonomy with overrides)
neg['adduct_cat'] = neg['adduct'].apply(classify_adduct)
neg['is_isf_adduct'] = (neg['adduct_cat']=='isf').astype(int)
neg['is_dubious_adduct'] = (neg['adduct_cat']=='dubious').astype(int)
neg['name_lower'] = neg['name'].fillna('').str.strip().str.lower()
has_ok_n = (neg[neg['adduct_cat']=='ok'].groupby('name_lower')['wiki_id'].count().rename('_has_ok'))
neg = neg.merge(has_ok_n.reset_index(), on='name_lower', how='left')
neg['has_ok_adduct'] = neg['_has_ok'].fillna(0).clip(upper=1).astype(int)
neg = neg.drop(columns='_has_ok')
neg['isf_no_mh'] = ((neg['is_isf_adduct']==1) & (neg['has_ok_adduct']==0)).astype(int)
n_add_n = neg.groupby('name_lower')['adduct'].nunique().rename('n_compound_adducts')
neg = neg.merge(n_add_n.reset_index(), on='name_lower', how='left')
neg['is_nacetyl'] = neg['name'].fillna('').str.contains(
    r'(?i)^N-?acetyl|^N\d-acetyl|^acetyl-.*(?:amine|alanine|valine|leucine|'
    r'isoleucine|glycine|serine|threonine|cysteine|methionine|phenylalanine|'
    r'tyrosine|tryptophan|aspart|glutam|histid|lysine|arginine|proline|ornithine|'
    r'citrulline|carnosine)'
).astype(int)

print(f'Neg annotated: {len(neg):,}')
print(neg['tier'].value_counts().to_string())

In [ ]:
# Load neg hits + compute neg features (same pipeline)
neg_hits_raw = pd.read_csv(NEG_HITS_CSV, low_memory=False)
neg_hits_raw = neg_hits_raw[neg_hits_raw['hit_source']=='reference'].copy()
neg_hits_raw['hit_ik14'] = neg_hits_raw['smiles'].fillna('').apply(get_ik14)
neg['anno_ik14'] = neg['smiles'].fillna('').apply(get_ik14)
with open(INCHIKEY_CACHE, 'w') as f:
    json.dump(ik_cache, f)

neg_hits_raw['entropy_similarity'] = pd.to_numeric(neg_hits_raw['entropy_similarity'], errors='coerce')
neg_info = neg[['wiki_id','anno_ik14','label','tier']].copy()
neg_joint = neg_hits_raw.merge(neg_info, on='wiki_id', how='inner')
neg_joint['is_anno_hit'] = (neg_joint['hit_ik14']!='') & (neg_joint['anno_ik14']!='') & (neg_joint['hit_ik14']==neg_joint['anno_ik14'])
neg_joint['dedup_key'] = neg_joint.apply(
    lambda r: (r['wiki_id'], r['hit_ik14']) if r['hit_ik14'] else (r['wiki_id'], r['lib_name']), axis=1)
neg_joint = neg_joint.sort_values('entropy_similarity', ascending=False).drop_duplicates('dedup_key').reset_index(drop=True)
print(f'Neg joint: {len(neg_joint):,}')

# Load neg query peaks
with open(QUERY_PEAKS) as f:
    neg_query_peaks = json.load(f)
print(f'Neg query peaks: {len(neg_query_peaks):,}')

In [ ]:
# Compute neg annotation features
t0 = time.time()
rows = []
for wid, g in neg_joint.groupby('wiki_id'):
    anno = g[g['is_anno_hit']]
    others = g[~g['is_anno_hit']]
    if len(anno) > 0:
        best = anno.loc[anno['entropy_similarity'].idxmax()]
        next_sim = others['entropy_similarity'].max() if len(others) > 0 else 0.0
        anno_rank = int((g['entropy_similarity'] >= best['entropy_similarity']).sum())
        lib_pk = lib_peaks_cache.get(best.get('library_wiki_id',''))
        qry_pk = neg_query_peaks.get(wid)
        sc = compute_scores(qry_pk, lib_pk)
        lib_mz = pd.to_numeric(best.get('lib_precursor_mz'), errors='coerce')
        obs_mz = neg.loc[neg['wiki_id']==wid,'precursor_mz'].iloc[0]
        row = {'wiki_id': wid,
               'anno_entropy_sim': best['entropy_similarity'],
               'anno_delta_mda': abs(obs_mz-lib_mz)*1000 if pd.notna(lib_mz) and lib_mz>0 else np.nan,
               'sim_gap': max(0.0, best['entropy_similarity']-next_sim) if pd.notna(next_sim) else best['entropy_similarity'],
               'anno_rank': anno_rank}
        if sc:
            row['anno_reverse']=sc['reverse_score']; row['anno_forward']=sc['forward_score']; row['max_deviation']=sc['max_deviation']
        else:
            row['anno_reverse']=np.nan; row['anno_forward']=np.nan; row['max_deviation']=np.nan
    else:
        row = dict.fromkeys(['anno_entropy_sim','anno_delta_mda','sim_gap','anno_rank',
                              'anno_reverse','anno_forward','max_deviation'], np.nan)
        row['wiki_id']=wid
    rows.append(row)
neg_anno_df = pd.DataFrame(rows)
print(f'Neg feature compute: {time.time()-t0:.1f}s')

neg_ev = neg[['wiki_id','name','label','tier','anno_ik14','precursor_mz',
               'is_isf_adduct','is_dubious_adduct','has_ok_adduct','isf_no_mh',
               'n_compound_adducts','is_nacetyl','adduct']].copy()
neg_ev['spectral_entropy'] = pd.to_numeric(neg['entropy'], errors='coerce')
neg_ev['delta_rt_abs'] = pd.to_numeric(neg['anno_delta_rt'], errors='coerce').abs()
neg_ev['identity_score'] = pd.to_numeric(neg['identity_score'], errors='coerce')
neg_ev = neg_ev.merge(neg_anno_df, on='wiki_id', how='left')
print(f'Neg evidence: {len(neg_ev):,}')

In [ ]:
# ── Train GBM on neg (regular_tp + fp) ──
FEATURE_COLS = [
    'delta_rt_abs','spectral_entropy','anno_entropy_sim','anno_forward','anno_reverse',
    'anno_delta_mda','sim_gap','anno_rank','max_deviation',
    'is_isf_adduct','isf_no_mh','is_dubious_adduct','n_compound_adducts','is_nacetyl',
]
print(f'Features: {len(FEATURE_COLS)}')

train_mask = neg_ev['tier'].isin(['regular_tp','fp'])
df_train = neg_ev[train_mask].copy()
y_train = (df_train['tier']=='regular_tp').astype(int)
train_medians = df_train[FEATURE_COLS].median()
X_train = df_train[FEATURE_COLS].fillna(train_medians).values

print(f'Train: regular_tp={int(y_train.sum())}, fp={int((~y_train.astype(bool)).sum())}')

gb = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_probs = cross_val_predict(gb, X_train, y_train, cv=skf, method='predict_proba')[:,1]
cv_auc = roc_auc_score(y_train, cv_probs)
print(f'5-fold CV AUC (neg): {cv_auc:.3f}  (target from v2: 0.851)')

gb.fit(X_train, y_train)

# Feature importance
print('\nFeature importance:')
for col, imp in sorted(zip(FEATURE_COLS, gb.feature_importances_), key=lambda x: -x[1]):
    print(f'  {col:25s}  {imp:.3f}')

## 7. Apply Neg-Trained GBM to Pos

In [ ]:
# Apply to pos
X_pos = ev[FEATURE_COLS].fillna(train_medians).values  # use neg train medians for imputation
y_pos = (ev['tier']=='regular_tp').astype(int)

pos_scores = gb.predict_proba(X_pos)[:,1]
ev['confidence_score'] = pos_scores

# AUC: pos TP vs FP
auc_pos = roc_auc_score(y_pos, pos_scores)
print(f'Pos AUC (TP={y_pos.sum()} vs FP={(~y_pos.astype(bool)).sum()}): {auc_pos:.3f}')
print(f'Neg CV AUC (reference): {cv_auc:.3f}')
print(f'Δ (pos - neg): {auc_pos - cv_auc:+.3f}')

# Score distributions
tp_scores = pos_scores[y_pos.astype(bool)]
fp_scores = pos_scores[~y_pos.astype(bool)]
print(f'\nScore distribution:')
print(f'  TP: mean={tp_scores.mean():.3f}, median={np.median(tp_scores):.3f}')
print(f'  FP: mean={fp_scores.mean():.3f}, median={np.median(fp_scores):.3f}')

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Score distribution TP vs FP
axes[0].hist(tp_scores, bins=40, alpha=0.6, color='steelblue', label=f'TP (n={len(tp_scores)})')
axes[0].hist(fp_scores, bins=40, alpha=0.6, color='salmon', label=f'FP (n={len(fp_scores)})')
axes[0].set_xlabel('Confidence score'); axes[0].set_ylabel('Count')
axes[0].set_title(f'Pos score distribution (AUC = {auc_pos:.3f})')
axes[0].legend()

# Neg vs pos CV probabilities on same axis
neg_tp_probs = cv_probs[y_train.astype(bool)]
neg_fp_probs = cv_probs[~y_train.astype(bool)]
axes[1].hist(neg_tp_probs, bins=30, alpha=0.4, color='navy', label=f'Neg TP')
axes[1].hist(neg_fp_probs, bins=30, alpha=0.4, color='darkred', label=f'Neg FP')
axes[1].hist(tp_scores, bins=30, alpha=0.4, color='steelblue', label=f'Pos TP')
axes[1].hist(fp_scores, bins=30, alpha=0.4, color='salmon', label=f'Pos FP')
axes[1].set_xlabel('GBM score'); axes[1].set_ylabel('Count')
axes[1].set_title('Score distributions — pos vs neg (normalized)')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. Export for Oliver

In [ ]:
# Output Excel with original pos row info + confidence score
out = pos.merge(
    ev[['wiki_id','confidence_score','has_anno_hit','anno_entropy_sim','anno_forward',
        'anno_reverse','sim_gap','anno_rank','max_deviation','anno_delta_mda',
        'is_isf_adduct','isf_no_mh','is_dubious_adduct','n_compound_adducts','is_nacetyl']],
    on='wiki_id', how='left'
).sort_values('confidence_score', ascending=False, na_position='last')

xlsx = f'{OUT_DIR}/pos_gbm_scores.xlsx'
csv = f'{OUT_DIR}/pos_gbm_scores.csv'
out.to_excel(xlsx, index=False)
out.to_csv(csv, index=False)

print(f'Saved: {xlsx}')
print(f'       {csv}')
print(f'\n{len(out)} rows, {len(out.columns)} cols')
print(f'\nScore summary by label:')
print(out.groupby('label')['confidence_score'].describe()[['count','mean','50%','std']].round(3).to_string())

## Summary

**Performance:**
- Pos AUC (TP vs FP): reported in cell 7
- Neg CV AUC (reference, 5-fold): should match v2's 0.851

**If pos AUC < neg CV AUC by >0.05:** covariate shift is hurting generalization. Options: retrain on pos, or joint train.

**If pos AUC ≈ neg CV AUC:** model generalizes — ship for Oliver's curation workflow.

**Output for Oliver:** `results/orbitrap_pos_gbm/pos_gbm_scores.xlsx`, sorted by confidence descending. He can scan top-confidence annotations first (trust them), scan bottom for suspect ones.